# 01 — Temperature & Salinity Contour Plots

A **contour (Hovmöller) plot** is one of the most useful tools in physical oceanography.  
The x-axis is **time**, the y-axis is **depth**, and the colour shows the value of a
variable (e.g. temperature or salinity).  This lets you see seasonal cycles, mixing
events, and the movement of water masses all in one plot.

In this notebook you will:
1. Download all profiles for a single float
2. Reshape the data into a 2-D grid (depth × time)
3. Plot temperature and salinity contour plots

**Prerequisite:** Complete `00_getting_started.ipynb` first.

In [ ]:
# --- Imports ---
from argopy import DataFetcher as ArgoDataFetcher

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy.interpolate import griddata      # For filling gaps in the depth grid

## 1 — Download float data

In [ ]:
# ---------------------------------------------------------------
# Choose your float — replace this WMO number to use a different float
# ---------------------------------------------------------------
WMO = 6903547

# Download data from the Argo ERDDAP server
print(f'Downloading data for float {WMO} ...')
loader = ArgoDataFetcher(src='erddap').float(WMO)

# argopy returns an xarray Dataset — we convert it to a pandas DataFrame
# which is easier to work with if you are new to Python
ds  = loader.to_xarray()       # xarray Dataset
df  = ds.to_dataframe().reset_index()   # pandas DataFrame

print(f'Download complete!  {len(df):,} data points retrieved.')

## 2 — Clean the data

Raw Argo data sometimes contains NaN values (missing measurements)  We remove them before plotting.

In [ ]:
# Drop rows that are missing a temperature, salinity, or pressure value
df = df.dropna(subset=['TEMP', 'PSAL', 'PRES'])

print(f'Rows after cleaning: {len(df):,}')

## 3 — Build the contour grid

A contour plot needs data on a regular 2-D grid.  We create one using `pandas.pivot_table`,
which averages all measurements at the same (cycle, pressure) pair.

We then define a regular pressure axis and interpolate onto it so the contours look smooth.

In [ ]:
# ---------------------------------------------------------------
# Parameters you can adjust
# ---------------------------------------------------------------
PRES_MIN   =   0    # Shallow end (dbar)
PRES_MAX   = 500    # Deep end — reduce to 200 to focus on the surface layer
PRES_STEP  =   5    # Vertical resolution of the output grid (dbar)
# ---------------------------------------------------------------

# Get a sorted list of cycle numbers and corresponding dates
# We use the median date of each cycle as the time axis value
cycle_times = (
    df.groupby('CYCLE_NUMBER')['TIME']
    .median()
    .reset_index()
    .rename(columns={'TIME': 'DATE'})
)

# Merge the date back onto the main DataFrame
df = df.merge(cycle_times, on='CYCLE_NUMBER')

# Regular pressure axis
pres_grid = np.arange(PRES_MIN, PRES_MAX + PRES_STEP, PRES_STEP)


def interpolate_profiles(df, variable, pres_grid):
    """
    Interpolate a variable onto a regular pressure grid for every cycle.

    Returns
    -------
    grid_data  : 2-D numpy array  shape (len(pres_grid), n_cycles)
    dates      : 1-D array of datetime objects for each cycle (x-axis)
    """
    cycles = sorted(df['CYCLE_NUMBER'].unique())
    dates  = []
    grid_data = np.full((len(pres_grid), len(cycles)), np.nan)

    for col_idx, cycle in enumerate(cycles):
        profile = (
            df[df['CYCLE_NUMBER'] == cycle]
            .dropna(subset=[variable, 'PRES'])
            .sort_values('PRES')
        )
        if len(profile) < 3:
            continue   # Skip cycles with too few points

        # Linear interpolation onto the regular pressure axis
        interpolated = np.interp(
            pres_grid,
            profile['PRES'].values,
            profile[variable].values,
            left=np.nan, right=np.nan
        )
        grid_data[:, col_idx] = interpolated
        dates.append(profile['DATE'].iloc[0])

    return grid_data, np.array(dates)


# Build the 2-D grids for temperature and salinity
temp_grid, dates = interpolate_profiles(df, 'TEMP', pres_grid)
psal_grid, _     = interpolate_profiles(df, 'PSAL', pres_grid)

print(f'Grid shape (depth × time): {temp_grid.shape}')

## 4 — Temperature contour plot

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))

# Filled contour plot
cf = ax.contourf(
    dates,          # x-axis: time
    pres_grid,      # y-axis: pressure
    temp_grid,      # colour: temperature
    levels=20,
    cmap='RdYlBu_r'
)

# Overlay black contour lines for clarity
cs = ax.contour(
    dates, pres_grid, temp_grid,
    levels=10,
    colors='k', linewidths=0.5, alpha=0.4
)
ax.clabel(cs, inline=True, fontsize=7, fmt='%g')

# Formatting
ax.invert_yaxis()                                     # Depth increases downward
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b\n%Y'))  # Month + year labels
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))   # Tick every 3 months
plt.setp(ax.get_xticklabels(), fontsize=9)

ax.set_ylabel('Pressure (dbar)', fontsize=12)
ax.set_title(f'Float {WMO} — Temperature (°C)', fontsize=14)

cbar = fig.colorbar(cf, ax=ax, pad=0.02)
cbar.set_label('Temperature (°C)', fontsize=11)

ax.grid(True, linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

## 5 — Salinity contour plot

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))

cf = ax.contourf(
    dates, pres_grid, psal_grid,
    levels=20,
    cmap='viridis'
)

cs = ax.contour(
    dates, pres_grid, psal_grid,
    levels=10,
    colors='k', linewidths=0.5, alpha=0.4
)
ax.clabel(cs, inline=True, fontsize=7, fmt='%.2f')

ax.invert_yaxis()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b\n%Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.setp(ax.get_xticklabels(), fontsize=9)

ax.set_ylabel('Pressure (dbar)', fontsize=12)
ax.set_title(f'Float {WMO} — Salinity (PSU)', fontsize=14)

cbar = fig.colorbar(cf, ax=ax, pad=0.02)
cbar.set_label('Salinity (PSU)', fontsize=11)

ax.grid(True, linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

## 6 — Combined figure (Temperature + Salinity, stacked)

Putting both plots in one figure makes it easy to compare the two variables.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 9), sharex=True)

# --- Temperature (top panel) ---
cf0 = axes[0].contourf(dates, pres_grid, temp_grid, levels=20, cmap='RdYlBu_r')
axes[0].contour(dates, pres_grid, temp_grid, levels=10,
                colors='k', linewidths=0.4, alpha=0.4)
axes[0].invert_yaxis()
axes[0].set_ylabel('Pressure (dbar)', fontsize=11)
axes[0].set_title(f'Float {WMO} — Temperature (°C)', fontsize=12)
fig.colorbar(cf0, ax=axes[0], pad=0.02, label='°C')

# --- Salinity (bottom panel) ---
cf1 = axes[1].contourf(dates, pres_grid, psal_grid, levels=20, cmap='viridis')
axes[1].contour(dates, pres_grid, psal_grid, levels=10,
                colors='k', linewidths=0.4, alpha=0.4)
axes[1].invert_yaxis()
axes[1].set_ylabel('Pressure (dbar)', fontsize=11)
axes[1].set_title(f'Float {WMO} — Salinity (PSU)', fontsize=12)
fig.colorbar(cf1, ax=axes[1], pad=0.02, label='PSU')

# Shared x-axis formatting
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%b\n%Y'))
axes[1].xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.setp(axes[1].get_xticklabels(), fontsize=9)

for ax in axes:
    ax.grid(True, linestyle='--', alpha=0.3)

plt.tight_layout()
plt.savefig('float_contours.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved to float_contours.png')

---

## Exercises

Try these to deepen your understanding:

1. **Change the depth range** — Edit `PRES_MAX` to zoom into just the upper 200 dbar (the surface layer).
2. **Change the colourmap** — Try `cmap='coolwarm'` or `cmap='plasma'` for temperature.
3. **Different float** — Change `WMO` to a float in the tropics and compare the temperature structure.
4. **Number of contour levels** — Change `levels=20` to `levels=10` or `levels=30`. How does this affect readability?

**Next notebook:** `02_mixed_layer_depth.ipynb`